<a href="https://colab.research.google.com/github/Radhakuchekar/Preparation/blob/pyspark/Tweets_Rolling_Averages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
from pyspark.sql.functions import *
spark

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType


# Define schema for the DataFrame
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("tweet_date", StringType(), True),
    StructField("tweet_count", IntegerType(), True)
])

# Create data as a list of tuples
data = [
    (111, "2022-06-01 00:00:00", 2),
    (111, "2022-06-02 00:00:00", 1),
    (111, "2022-06-03 00:00:00", 3),
    (111, "2022-06-04 00:00:00", 4),
    (111, "2022-06-05 00:00:00", 5)
]

# Create DataFrame
tweets_df = spark.createDataFrame(data, schema=schema)
tweets_df = tweets_df.withColumn("tweet_date", to_timestamp(col("tweet_date"), "yyyy-MM-dd HH:mm:ss"))

# Show the DataFrame
tweets_df.show()


+-------+-------------------+-----------+
|user_id|         tweet_date|tweet_count|
+-------+-------------------+-----------+
|    111|2022-06-01 00:00:00|          2|
|    111|2022-06-02 00:00:00|          1|
|    111|2022-06-03 00:00:00|          3|
|    111|2022-06-04 00:00:00|          4|
|    111|2022-06-05 00:00:00|          5|
+-------+-------------------+-----------+



In [5]:
# Given a table of tweet data over a specified time period, calculate the 3-day rolling average of tweets for each user.
#  Output the user ID, tweet date, and rolling averages rounded to 2 decimal places.

In [9]:
from pyspark.sql.window import Window
# for scala import org.apache.sql.expressions.Window

In [15]:
window_spec = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(-2, 0)

In [20]:
tweets_df.withColumn("movingAvg", round(avg("tweet_count").over(window_spec),2))

user_id,tweet_date,tweet_count,movingAvg
111,2022-06-01 00:00:00,2,2.0
111,2022-06-02 00:00:00,1,1.5
111,2022-06-03 00:00:00,3,2.0
111,2022-06-04 00:00:00,4,2.67
111,2022-06-05 00:00:00,5,4.0


In [ ]:
# Row-Based Rolling Window
# Unbounded preceding (from the beginning to the current row)
window_spec_cumulative = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(Window.unboundedPreceding, 0)
# Unbounded preceding and following (entire partition)
window_spec_full = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
# Rolling window of the current row and 2 preceding rows
window_spec_rolling = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(-2, 0)

# Time-Based Rolling Window
window_spec_time_rolling = Window.partitionBy("user_id").orderBy("timestamp").rangeBetween(-3 * 86400, 0)  # 3 days in seconds
# Current row and 2 following rows
window_spec_future = Window.partitionBy("user_id").orderBy("tweet_date").rowsBetween(0, 2)

